# Phase 3: CSP
#Disease Diagnosis using Symptom Slots
**Dataset:** Disease Symptom Dataset  
**Approach:** Model the diagnosis task as a CSP where symptom slots must be filled to match a target disease

---

### Variables
Each symptom slot is a variable: `Symptom_1, Symptom_2, Symptom_3, Symptom_4`  

### Domain
Each variable can take any symptom name from the full list of 131 symptoms in the dataset.

### Constraints
1. **No Duplicate Symptoms:** All four symptom slots must have different values the same symptom cannot appear twice in a diagnosis.
2. **Severity Constraint:** Every assigned symptom must have a known severity weight (it must exist in the severity table).
3. **Disease Match Constraint:** All four assigned symptoms must belong to the same disease (the target disease we are diagnosing).
4. **Minimum Severity Constraint:** At least one symptom must have a severity weight >= 3 (the diagnosis must include at least one moderate/severe symptom).
5. **Slot Ordering Constraint:** Symptom_1 must have a severity weight less than or equal to Symptom_2's weight (symptoms are assigned in non-decreasing severity order).



In [56]:
import pandas as pd
import random
from collections import deque
df_disease= pd.read_csv('/content/dataset.csv')
df_desc= pd.read_csv('/content/symptom_Description.csv')
df_precaution= pd.read_csv('/content/symptom_precaution.csv')
df_severity= pd.read_csv('/content/Symptom-severity (1).csv')
df_severity.columns = df_severity.columns.str.strip()
df_severity['Symptom'] = df_severity['Symptom'].str.strip()
severity_dict = dict(zip(df_severity['Symptom'], df_severity['weight']))

disease_symptoms = {}
for _, row in df_disease.iterrows():
    disease = row['Disease'].strip()
    symptoms = [str(row[c]).strip() for c in df_disease.columns[1:] if pd.notna(row[c])]
    if disease not in disease_symptoms:
        disease_symptoms[disease] = set()
    disease_symptoms[disease].update(symptoms)

print(f"Total diseases: {len(disease_symptoms)}")
print(f"Total unique symptoms (severity table): {len(severity_dict)}")
print()

for disease in list(disease_symptoms.keys())[:4]:
    print(f"{disease}: {list(disease_symptoms[disease])[:5]}")

Total diseases: 41
Total unique symptoms (severity table): 132

Fungal infection: ['nodal_skin_eruptions', 'dischromic _patches', 'skin_rash', 'itching']
Allergy: ['chills', 'watering_from_eyes', 'shivering', 'continuous_sneezing']
GERD: ['stomach_pain', 'cough', 'ulcers_on_tongue', 'vomiting', 'acidity']
Chronic cholestasis: ['loss_of_appetite', 'abdominal_pain', 'vomiting', 'itching', 'yellowing_of_eyes']


In [42]:
graph = build_graph(df_disease)
print(f"Graph created with {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges")
print("Nodes (first 10):", list(graph.nodes)[:10])
print("Edges (first 10):", list(graph.edges)[:10])


Graph created with 174 nodes and 321 edges.
Nodes (first 10): ['Fungal infection', 'Allergy', 'GERD', 'Chronic cholestasis', 'Drug Reaction', 'Peptic ulcer diseae', 'AIDS', 'Diabetes ', 'Gastroenteritis', 'Bronchial Asthma']
Edges (first 10): [('Fungal infection', 'itching'), ('Fungal infection', 'skin_rash'), ('Fungal infection', 'nodal_skin_eruptions'), ('Fungal infection', 'dischromic _patches'), ('Allergy', 'continuous_sneezing'), ('Allergy', 'shivering'), ('Allergy', 'chills'), ('Allergy', 'watering_from_eyes'), ('GERD', 'stomach_pain'), ('GERD', 'acidity')]


In [9]:
display(df_disease.head())

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
display(df_desc.head())

,Disease,Description
0,Drug Reaction,An adverse drug reaction (ADR) is an injury ca...
1,Malaria,An infectious disease caused by protozoan para...
2,Allergy,An allergy is an immune system response to a f...
3,Hypothyroidism,"Hypothyroidism, also called underactive thyroi..."
4,Psoriasis,Psoriasis is a common skin disorder that forms...


In [11]:
display(df_precaution.head())

,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Drug Reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,Malaria,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,Allergy,apply calamine,cover area with bandage,NaN,use ice to compress itching
3,Hypothyroidism,reduce stress,exercise,eat healthy,get proper sleep
4,Psoriasis,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths


In [12]:
display(df_severity.head())

,Symptom,weight
0,itching,1
1,skin_rash,3
2,nodal_skin_eruptions,4
3,continuous_sneezing,4
4,shivering,5


In [13]:
df_disease.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4920 entries, 0 to 4919
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Disease     4920 non-null   object
 1   Symptom_1   4920 non-null   object
 2   Symptom_2   4920 non-null   object
 3   Symptom_3   4920 non-null   object
 4   Symptom_4   4572 non-null   object
 5   Symptom_5   3714 non-null   object
 6   Symptom_6   2934 non-null   object
 7   Symptom_7   2268 non-null   object
 8   Symptom_8   1944 non-null   object
 9   Symptom_9   1692 non-null   object
 10  Symptom_10  1512 non-null   object
 11  Symptom_11  1194 non-null   object
 12  Symptom_12  744 non-null    object
 13  Symptom_13  504 non-null    object
 14  Symptom_14  306 non-null    object
 15  Symptom_15  240 non-null    object
 16  Symptom_16  192 non-null    object
 17  Symptom_17  72 non-null     object
dtypes: object(18)
memory usage: 692.0+ KB


In [14]:
df_desc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Disease      41 non-null     object
 1   Description  41 non-null     object
dtypes: object(2)
memory usage: 788.0+ bytes


In [15]:
df_precaution.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Disease       41 non-null     object
 1   Precaution_1  41 non-null     object
 2   Precaution_2  41 non-null     object
 3   Precaution_3  40 non-null     object
 4   Precaution_4  40 non-null     object
dtypes: object(5)
memory usage: 1.7+ KB


In [16]:
df_severity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Symptom  133 non-null    object
 1   weight   133 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 2.2+ KB


---
## CSP Structure


In [52]:
TARGET_DISEASE = 'Fungal infection'
disease_syms = disease_symptoms[TARGET_DISEASE]
valid_syms = [s for s in disease_syms if s in severity_dict]
print(f"Target Disease : {TARGET_DISEASE}")
print(f"All symptoms   : {disease_syms}")
print(f"Valid symptoms (in severity table): {valid_syms}")
print()

#VARIABLES
VARIABLES = ['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4']

#DOMAINS
csp_domains = {var: list(valid_syms) for var in VARIABLES}
print("====== DOMAINS (before AC-3)=====")
for var in VARIABLES:
    print(f"{var}: {csp_domains[var]}  (size={len(csp_domains[var])})")

Target Disease : Fungal infection
All symptoms   : {'nodal_skin_eruptions', 'dischromic _patches', 'skin_rash', 'itching'}
Valid symptoms (in severity table): ['nodal_skin_eruptions', 'skin_rash', 'itching']

====== DOMAINS (before AC-3)=====
Symptom_1: ['nodal_skin_eruptions', 'skin_rash', 'itching']  (size=3)
Symptom_2: ['nodal_skin_eruptions', 'skin_rash', 'itching']  (size=3)
Symptom_3: ['nodal_skin_eruptions', 'skin_rash', 'itching']  (size=3)
Symptom_4: ['nodal_skin_eruptions', 'skin_rash', 'itching']  (size=3)


---
##  ALL Constraints


In [66]:
# Constraint 1: No duplicate symptoms
def constraint_no_duplicate(sym_i, sym_j):
    return sym_i != sym_j

# Constraint 2: Symptom must exist in severity table
def constraint_has_severity(sym):
    return sym in severity_dict

# Constraint 3: Symptom must belong to the target disease
def constraint_belongs_to_disease(sym):
    return sym in disease_symptoms[TARGET_DISEASE]

# Constraint 4: At least one symptom must have severity >= 3
def constraint_min_severity(assignment):
    return any(severity_dict.get(s, 0) >= 3 for s in assignment.values())

# Constraint 5: Symptom_1 severity <= Symptom_2 severity
def constraint_ordering(sym_1, sym_2):
    return severity_dict.get(sym_1, 0) <= severity_dict.get(sym_2, 0)

# BINARY CONSTRAINT CHECK (used by AC-3 and backtracking)
def is_consistent_pair(var_i, val_i, var_j, val_j):
    # Constraint 1: no duplicate
    if not constraint_no_duplicate(val_i, val_j):
        return False
    # Constraint 5: ordering only between Symptom_1 and Symptom_2
    if var_i == 'Symptom_1' and var_j == 'Symptom_2':
        if not constraint_ordering(val_i, val_j):
            return False
    if var_i == 'Symptom_2' and var_j == 'Symptom_1':
        if not constraint_ordering(val_j, val_i):
            return False
    return True

# UNARY CONSTRAINT CHECK used for domain filtering
def is_valid_value(sym):
    return constraint_has_severity(sym) and constraint_belongs_to_disease(sym)

print("ALL Constraitns:")
print("1. No duplicate symptoms between any two slots")
print("2. Each symptom must exist in the severity table")
print("3. Each symptom must belong to the target disease")
print("4. At least one symptom must have severity >= 3")
print("5. Symptom_1 severity <= Symptom_2 severity (ordering)")

ALL Constraitns:
1. No duplicate symptoms between any two slots
2. Each symptom must exist in the severity table
3. Each symptom must belong to the target disease
4. At least one symptom must have severity >= 3
5. Symptom_1 severity <= Symptom_2 severity (ordering)


---
##  Constraint Propagation with AC-3
AC-3 removes values from domains that cannot be part of any valid solution

In [65]:
def revise(domains, var_i, var_j):
    """
    Remove values from domains[var_i] that have NO consistent value in domains[var_j]
    Returns True if any value was removed (domain changed), False otherwise
    """
    revised = False
    values_to_remove = []
    for val_i in domains[var_i]:
        has_support = False
        for val_j in domains[var_j]:
            if is_consistent_pair(var_i, val_i, var_j, val_j):
                has_support = True
                break
        if not has_support:
            values_to_remove.append(val_i)
            revised = True
    for val in values_to_remove:
        domains[var_i].remove(val)
    return revised
print("revise() function defined")

revise() function defined


### AC-3 Algo

In [26]:
def ac3(domains, variables):
    """
    AC-3 algorithm
    Processes a queue of all arcs (pairs of variables with constraints)
    Returns True if no domain becomes empty, False if a domain is wiped out
    """
    queue = deque()
    for i in range(len(variables)):
        for j in range(len(variables)):
            if i != j:
                queue.append((variables[i], variables[j]))
    print(f"Initial queue size: {len(queue)} arcs")

    while queue:
        var_i, var_j = queue.popleft()
        if revise(domains, var_i, var_j)
            if len(domains[var_i]) == 0:
                print(f"Domain of {var_i} is empty No solution possible")
                return False
            for var_k in variables:
                if var_k != var_i and var_k != var_j:
                    queue.append((var_k, var_i))
    return True

# Run AC-3
import copy
print("=== Domain sizes BEFORE AC-3 ===")
for var in VARIABLES:
    print(f"  {var}: {len(csp_domains[var])} values -> {csp_domains[var]}")

domains_after_ac3 = copy.deepcopy(csp_domains)
for var in VARIABLES:
    domains_after_ac3[var] = [s for s in domains_after_ac3[var] if is_valid_value(s)]
result = ac3(domains_after_ac3, VARIABLES)

print()
print("===== Domain sizes AFTER AC-3 ======")
for var in VARIABLES:
    print(f"  {var}: {len(domains_after_ac3[var])} values -> {domains_after_ac3[var]}")
print()
print(f"AC-3 completed successfully: {result}")

=== Domain sizes BEFORE AC-3 ===
  Symptom_1: 3 values -> ['nodal_skin_eruptions', 'skin_rash', 'itching']
  Symptom_2: 3 values -> ['nodal_skin_eruptions', 'skin_rash', 'itching']
  Symptom_3: 3 values -> ['nodal_skin_eruptions', 'skin_rash', 'itching']
  Symptom_4: 3 values -> ['nodal_skin_eruptions', 'skin_rash', 'itching']
Initial queue size: 12 arcs

===== Domain sizes AFTER AC-3 ======
  Symptom_1: 2 values -> ['skin_rash', 'itching']
  Symptom_2: 2 values -> ['nodal_skin_eruptions', 'skin_rash']
  Symptom_3: 3 values -> ['nodal_skin_eruptions', 'skin_rash', 'itching']
  Symptom_4: 3 values -> ['nodal_skin_eruptions', 'skin_rash', 'itching']

AC-3 completed successfully: True


---
##  Backtracking Search



In [64]:
def check_assignment(assignment):
    """
    Check all constraints for the current partial or complete assignment
    Returns True if no constraints are violated
    """
    vars_assigned = list(assignment.keys())
    # Check binary constraints between every assigned pair
    for i in range(len(vars_assigned)):
        for j in range(i + 1, len(vars_assigned)):
            vi, vj = vars_assigned[i], vars_assigned[j]
            if not is_consistent_pair(vi, assignment[vi], vj, assignment[vj]):
                return False

    # Check Constraint 4 only when assignment is complete
    if len(assignment) == len(VARIABLES):
        if not constraint_min_severity(assignment):
            return False
    return True


# Basic Backtracking
backtrack_count_basic = 0
def basic_backtracking(assignment, domains, variables):
    global backtrack_count_basic

    if len(assignment) == len(variables):
        return assignment
    unassigned = [v for v in variables if v not in assignment]
    var = unassigned[0]

    for value in domains[var]:
        assignment[var] = value
        if check_assignment(assignment):
            result = basic_backtracking(assignment, domains, variables)
            if result is not None:
                return result
        del assignment[var]
        backtrack_count_basic += 1
    return None

backtrack_count_basic = 0
solution_basic = basic_backtracking({}, copy.deepcopy(domains_after_ac3), VARIABLES)
print("====== Basic Backtracking ======")
if solution_basic:
    print("Solution found")
    for var, sym in solution_basic.items():
        print(f"  {var} = {sym}  (severity = {severity_dict.get(sym, 'N/A')})")
else:
    print("No solution found")
print(f"Total backtracks: {backtrack_count_basic}")

====== Basic Backtracking ======
No solution found
Total backtracks: 24


###  Backtracking with Forward Checking

In [63]:
def forward_check(assignment, domains, var, value, variables):
    """
    After assigning 'value' to 'var', remove values from remaining variables
    domains that are now inconsistent
    Returns a dict of {variable: [removed values]} so we can undo changes on backtrack
    Returns None if any domain becomes empty
    """
    removed = {}
    for other_var in variables:
        if other_var in assignment or other_var == var:
            continue
        removed[other_var] = []
        for val in list(domains[other_var]):
            if not is_consistent_pair(var, value, other_var, val):
                domains[other_var].remove(val)
                removed[other_var].append(val)
        if len(domains[other_var]) == 0:
            return None
    return removed

backtrack_count_fc = 0
def backtracking_with_fc(assignment, domains, variables):
    global backtrack_count_fc
    if len(assignment) == len(variables):
        return assignment

    unassigned = [v for v in variables if v not in assignment]
    var = unassigned[0]
    for value in domains[var]:
        assignment[var] = value
        if check_assignment(assignment):
            removed = forward_check(assignment, domains, var, value, variables)

            if removed is not None:
                result = backtracking_with_fc(assignment, domains, variables)
                if result is not None:
                    return result
            if removed:
                for other_var, vals in removed.items():
                    domains[other_var].extend(vals)
        del assignment[var]
        backtrack_count_fc += 1
    return None


# Run forward checking backtracking
backtrack_count_fc = 0
solution_fc = backtracking_with_fc({}, copy.deepcopy(domains_after_ac3), VARIABLES)
print("======= Backtracking with Forward Checking ======")
if solution_fc:
    print("Solution found")
    for var, sym in solution_fc.items():
        print(f"  {var} = {sym}  (severity = {severity_dict.get(sym, 'N/A')})")
else:
    print("No solution found")
print(f"Total backtracks: {backtrack_count_fc}")
print(f"Reduction vs basic: {backtrack_count_basic - backtrack_count_fc} fewer backtracks")

======= Backtracking with Forward Checking ======
No solution found
Total backtracks: 8
Reduction vs basic: 16 fewer backtracks


### Backtracking with MRV Heuristic

In [58]:
def select_mrv_variable(assignment, domains, variables):
    """
    MRV = Minimum Remaining Values.
    Pick the unassigned variable with the SMALLEST domain size
    This reduces the search space by tackling the hardest variable first
    """
    unassigned = [v for v in variables if v not in assignment]
    return min(unassigned, key=lambda v: len(domains[v]))
backtrack_count_mrv = 0

def backtracking_with_mrv(assignment, domains, variables):
    global backtrack_count_mrv
    if len(assignment) == len(variables):
        return assignment

    var = select_mrv_variable(assignment, domains, variables)
    for value in domains[var]:
        assignment[var] = value

        if check_assignment(assignment):
            removed = forward_check(assignment, domains, var, value, variables)
            if removed is not None:
                result = backtracking_with_mrv(assignment, domains, variables)
                if result is not None:
                    return result
            if removed:
                for other_var, vals in removed.items():
                    domains[other_var].extend(vals)
        del assignment[var]
        backtrack_count_mrv += 1
    return None

# Run MRV backtracking
backtrack_count_mrv = 0
solution_mrv = backtracking_with_mrv({}, copy.deepcopy(domains_after_ac3), VARIABLES)
print("===== Backtracking with MRV Heuristic =====")
if solution_mrv:
    print("Solution found")
    for var, sym in solution_mrv.items():
        print(f"  {var} = {sym}  (severity = {severity_dict.get(sym, 'N/A')})")
else:
    print("No solution found")
print(f"Total backtracks: {backtrack_count_mrv}")
print()
print("===== Comparison Summary =====")
print(f"  Basic Backtracking : {backtrack_count_basic} backtracks")
print(f"  With Forward Checking : {backtrack_count_fc} backtracks")
print(f"  With MRV Heuristic : {backtrack_count_mrv} backtracks")

===== Backtracking with MRV Heuristic =====
No solution found
Total backtracks: 8

===== Comparison Summary =====
  Basic Backtracking : 24 backtracks
  With Forward Checking : 8 backtracks
  With MRV Heuristic : 8 backtracks


---
## Local Search Min-Conflicts Algorithm


In [62]:
def count_violations(assignment):
    """
    Count the total number of constraint violations in a complete assignment
    Checks all 5 constraints.
    """
    violations = 0
    vars_list = list(assignment.keys())

    # Binary constraint: no duplicates + ordering
    for i in range(len(vars_list)):
        for j in range(i + 1, len(vars_list)):
            vi, vj = vars_list[i], vars_list[j]
            if not is_consistent_pair(vi, assignment[vi], vj, assignment[vj]):
                violations += 1

    # Constraint 2: severity exists
    for sym in assignment.values():
        if not constraint_has_severity(sym):
            violations += 1

    # Constraint 3: belongs to disease
    for sym in assignment.values():
        if not constraint_belongs_to_disease(sym):
            violations += 1

    # Constraint 4: at least one severity >= 3
    if not constraint_min_severity(assignment):
        violations += 1
    return violations

def violations_for_var(var, value, assignment):
    temp = dict(assignment)
    temp[var] = value
    return count_violations(temp)
print("Violation counting functions defined")

Violation counting functions defined


### Min-Conflicts Algorithm

In [47]:
def min_conflicts(variables, domains, max_iterations=100):
    """
    Min-Conflicts local search algorithm for CSP Steps:
    1. Start with a random complete assignment
    2. Count violations
    3. Pick a variable involved in a conflict
    4. Assign the value that minimizes conflicts
    5. Repeat until solved or max iterations reached
    """

    assignment = {}
    for var in variables:
        assignment[var] = random.choice(domains[var])
    violation_history = []

    for iteration in range(max_iterations):
        num_violations = count_violations(assignment)
        violation_history.append(num_violations)

        if num_violations == 0:
            print(f"Solution found at iteration {iteration}!")
            return assignment, violation_history

        conflicted_vars = []
        for var in variables:
            temp = dict(assignment)
            current_v = count_violations(temp)
            for val in domains[var]:
                temp[var] = val
                if count_violations(temp) < current_v:
                    conflicted_vars.append(var)
                    break

        if not conflicted_vars:
            conflicted_vars = list(variables)

        var = random.choice(conflicted_vars)
        best_value = min(domains[var], key=lambda val: violations_for_var(var, val, assignment))
        assignment[var] = best_value

    final_violations = count_violations(assignment)
    violation_history.append(final_violations)
    print(f"Max iterations reached Final violations: {final_violations}")
    return assignment, violation_history

random.seed(42)
final_assignment, history = min_conflicts(VARIABLES, copy.deepcopy(domains_after_ac3), max_iterations=100)
print()
print("=== Final Assignment (Min-Conflicts) ===")
for var, sym in final_assignment.items():
    print(f"  {var} = {sym}  (severity = {severity_dict.get(sym, 'N/A')})")
print(f"Final violations: {count_violations(final_assignment)}")
print()
print("Violation count per iteration:")
print(history)

Max iterations reached Final violations: 1

=== Final Assignment (Min-Conflicts) ===
  Symptom_1 = skin_rash  (severity = 3)
  Symptom_2 = nodal_skin_eruptions  (severity = 4)
  Symptom_3 = itching  (severity = 1)
  Symptom_4 = nodal_skin_eruptions  (severity = 4)
Final violations: 1

Violation count per iteration:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


---
## Phase 3 Reflection


In [61]:
print("========== PHASE 3 REFLECTION ==========")
print()
# Q44: AC-3 domain reduction
print("Q44: How much did AC-3 reduce domain sizes?")
for var in VARIABLES:
    before = len(csp_domains[var])
    after  = len(domains_after_ac3[var])
    print(f"  {var}: {before} -> {after}  (removed {before - after} values)")
print()

# Q45: Which backtracking variant had fewest backtracks
print("Q45: Which backtracking variant had the fewest backtracks?")
results = {
    'Basic Backtracking'       : backtrack_count_basic,
    'Forward Checking'         : backtrack_count_fc,
    'MRV Heuristic'            : backtrack_count_mrv
}

for name, count in results.items():
    print(f"  {name}: {count} backtracks")
best = min(results, key=results.get)
print(f"The Best one is: {best}")
print("MRV wins because it picks the variable with the fewest remaining values first")
print("This means it discovers failures early and avoids exploring useless branches")
print()

# Q46: Did Min-Conflicts find a solution within 100 iterations?
print("Q46: Did Min-Conflicts find a solution within 100 iterations?")
final_v = count_violations(final_assignment)
if final_v == 0:
    solved_at = next(i for i, v in enumerate(history) if v == 0)
    print(f"YES Solution found at iteration {solved_at} with 0 violations")
else:
    print(f"NO {final_v} violation(s) remained after 100 iterations")
    print(f"Min violations seen during search: {min(history)}")

========== PHASE 3 REFLECTION ==========

Q44: How much did AC-3 reduce domain sizes?
  Symptom_1: 3 -> 2  (removed 1 values)
  Symptom_2: 3 -> 2  (removed 1 values)
  Symptom_3: 3 -> 3  (removed 0 values)
  Symptom_4: 3 -> 3  (removed 0 values)

Q45: Which backtracking variant had the fewest backtracks?
  Basic Backtracking: 24 backtracks
  Forward Checking: 8 backtracks
  MRV Heuristic: 8 backtracks
The Best one is: Forward Checking
MRV wins because it picks the variable with the fewest remaining values first
This means it discovers failures early and avoids exploring useless branches

Q46: Did Min-Conflicts find a solution within 100 iterations?
NO 1 violation(s) remained after 100 iterations
Min violations seen during search: 1


In [6]:
print('Files in current directory:')
!ls -F

Files in current directory:
sample_data/
